# Lab04.3 — HSV and RGB Color Detection

**Objective:** detect color regions in images captured with Logitech C270. This notebook combines two experiments: (1) filtering a red marker against skin-colored background, and (2) detecting red, green and blue regions simultaneously.

For 8-bit images OpenCV uses **H = 0–179**, **S = 0–255**, **V = 0–255**. HSV limits are supplemented with RGB-channel dominance to reduce false positives.

## 1. Camera setup

Run once. Each experiment captures a fresh frame and releases the camera before display.

In [ ]:
import cv2
import numpy as np
import time
from contextlib import contextmanager
from IPython.display import display, Image

CAMERA_DEVICE = "/dev/video0"
FRAME_WIDTH, FRAME_HEIGHT = 640, 480
WARMUP_FRAMES = 30

@contextmanager
def camera_session():
    # Open the Logitech C270 through the Linux V4L2 interface.
    cap = cv2.VideoCapture(CAMERA_DEVICE, cv2.CAP_V4L2)
    try:
        if not cap.isOpened():
            raise RuntimeError("Cannot open camera: " + CAMERA_DEVICE)
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, FRAME_WIDTH)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, FRAME_HEIGHT)
        for _ in range(WARMUP_FRAMES):
            ok, frame = cap.read()
            if not ok or frame is None:
                raise RuntimeError("Camera warm-up frame failed.")
            time.sleep(0.05)
        yield cap
    finally:
        cap.release()
        print("Camera released.")

def capture_frame():
    with camera_session() as cap:
        ok, frame = cap.read()
        if not ok or frame is None:
            raise RuntimeError("No camera frame captured.")
    return frame

def show_bgr(img):
    ok, encoded = cv2.imencode(".jpg", img)
    if not ok:
        raise RuntimeError("JPEG encoding failed.")
    display(Image(data=encoded.tobytes()))

## 2. Detect one vertical red marker

Hold a vertical red marker in the camera image. Compare the original frame, red mask and annotated detection. The aspect-ratio criterion is deliberately specific to a vertical marker.

In [ ]:
frame = capture_frame()
hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
# Red hue wraps around 0 on OpenCV's 0...179 hue scale.
red_hsv = cv2.bitwise_or(
    cv2.inRange(hsv, (0, 100, 50), (10, 255, 255)),
    cv2.inRange(hsv, (170, 100, 50), (179, 255, 255))
)
# Convert channels to signed integers before applying differences.
b, g, r = [channel.astype(np.int16) for channel in cv2.split(frame)]
red_dominance = (r > g + 45) & (r > b + 45) & (r > 80)
red_mask = cv2.bitwise_and(red_hsv, red_dominance.astype(np.uint8) * 255)
kernel = np.ones((3, 3), np.uint8)
red_mask = cv2.morphologyEx(red_mask, cv2.MORPH_OPEN, kernel)
red_mask = cv2.morphologyEx(red_mask, cv2.MORPH_CLOSE, kernel)
result = frame.copy()
contours, _ = cv2.findContours(
    red_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
)
count = 0
for cnt in contours:
    if cv2.contourArea(cnt) < 80:
        continue
    x, y, w, h = cv2.boundingRect(cnt)
    # This filter is specific to a VERTICAL marker, not all red objects.
    if h / float(max(w, 1)) < 1.5:
        continue
    cv2.rectangle(result, (x, y), (x + w, y + h), (0, 0, 255), 2)
    cv2.putText(result, "RED", (x, max(15, y - 8)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
    count += 1
print("Red regions detected:", count)
red_panel = np.hstack([
    cv2.resize(im, (320, 240))
    for im in (frame, cv2.cvtColor(red_mask, cv2.COLOR_GRAY2BGR), result)
])
print("Original | Red mask | Detection")
show_bgr(red_panel)

## 3. Detect red, green and blue regions simultaneously

Present red, green and blue markers. HSV thresholds and channel-dominance margins differ by color. Morphological opening removes noise; closing fills small gaps. A marker with a white label can yield more than one contour.

In [ ]:
frame = capture_frame()
COLORS = {
    "RED": {"ranges": [((0, 100, 50), (10, 255, 255)),
                       ((170, 100, 50), (179, 255, 255))],
            "margin": 35, "box": (0, 0, 255)},
    "GREEN": {"ranges": [((35, 45, 35), (90, 255, 255))],
              "margin": 10, "box": (0, 255, 0)},
    "BLUE": {"ranges": [((95, 80, 50), (130, 255, 255))],
             "margin": 35, "box": (255, 0, 0)}
}
MIN_AREA = 150
hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
b, g, r = [channel.astype(np.int16) for channel in cv2.split(frame)]
result = frame.copy()
masks, counts = {}, {}
for name, cfg in COLORS.items():
    hsv_mask = np.zeros(hsv.shape[:2], dtype=np.uint8)
    for lower, upper in cfg["ranges"]:
        hsv_mask = cv2.bitwise_or(hsv_mask, cv2.inRange(hsv, lower, upper))
    margin = cfg["margin"]
    if name == "RED":
        dominant = (r > g + margin) & (r > b + margin) & (r > 80)
    elif name == "GREEN":
        dominant = (g > r + margin) & (g > b + margin) & (g > 50)
    else:
        dominant = (b > r + margin) & (b > g + margin) & (b > 80)
    mask = cv2.bitwise_and(hsv_mask, dominant.astype(np.uint8) * 255)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    mask = cv2.morphologyEx(
        mask, cv2.MORPH_CLOSE,
        cv2.getStructuringElement(cv2.MORPH_RECT, (7, 7))
    )
    masks[name] = mask
    contours, _ = cv2.findContours(
        mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )
    count = 0
    for cnt in contours:
        if cv2.contourArea(cnt) < MIN_AREA:
            continue
        x, y, w, h = cv2.boundingRect(cnt)
        moments = cv2.moments(cnt)
        if moments["m00"]:
            cx = int(moments["m10"] / moments["m00"])
            cy = int(moments["m01"] / moments["m00"])
        else:
            cx, cy = x + w // 2, y + h // 2
        cv2.rectangle(result, (x, y), (x + w, y + h), cfg["box"], 2)
        cv2.circle(result, (cx, cy), 4, cfg["box"], -1)
        cv2.putText(
            result, "{} ({},{})".format(name, cx, cy),
            (x, max(15, y - 8)), cv2.FONT_HERSHEY_SIMPLEX,
            0.45, cfg["box"], 1
        )
        count += 1
    counts[name] = count
print("Detection results:")
for name in ("RED", "GREEN", "BLUE"):
    print("{}: {}".format(name, counts[name]))
top = np.hstack([
    cv2.resize(im, (320, 240)) for im in (frame, result)
])
bottom = np.hstack([
    cv2.resize(cv2.cvtColor(masks[name], cv2.COLOR_GRAY2BGR), (320, 240))
    for name in ("RED", "GREEN", "BLUE")
])
print("Original | Detection")
show_bgr(top)
print("RED mask | GREEN mask | BLUE mask")
show_bgr(bottom)

## Student checks

1. Examine the masks and explain why HSV alone also selected parts of the hand in earlier tests.
2. Rotate the red marker to test the *vertical-marker* shape restriction in Section 2.
3. In Section 3, change GREEN `margin` from `10` to `35` and observe the effect.
4. Explain why contour count is not necessarily physical-object count.

**Observed during hardware trials:** the red-marker detector marked one vertical red region without obvious skin false positives. The simultaneous detector identified three marker colors with RED: 2, GREEN: 2, BLUE: 1 colored regions. These are observations for the tested lighting and markers, not guaranteed counts.